In [2]:
!pip install yfinance

     -------------------------------------- 104.7/104.7 kB 3.0 MB/s eta 0:00:00
  Using cached multitasking-0.0.11-py3-none-any.whl (8.5 kB)
     -------------------------------------- 948.2/948.2 kB 5.5 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached html5lib-1.1-py2.py3-none-any.whl (112 kB)
  Created wheel for peewee: filename=peewee-3.17.8-py3-none-any.whl size=139012 sha256=974a503d65570fcf7f8f0de1fc9c43abbc1e8e01f65b1273523560ae16ea132e
  Stored in directory: c:\users\scasny\appdata\local\pip\cache\wheels\e6\ad\a7\5999d65dc5f97b04e0d5daff474c56bb572083797d2f568d84
Successfully built peewee



[notice] A new release of pip available: 22.3.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [78]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint

# Step 1: Fetch 1-hour interval Bitcoin price data using yfinance
def fetch_bitcoin_data():
    """
    Fetches historical Bitcoin price data with a 1-hour interval.
    """
    ticker = "BTC-USD"
    start_date = "2023-01-01"  # Adjust start date as needed
    end_date = "2023-12-31"    # Adjust end date as needed
    interval = "1h"
    
    # Download data
    btc_data = yf.download(ticker, start=start_date, end=end_date, interval=interval)
    
    # Remove multi-level columns to keep only 'Close'
    if isinstance(btc_data.columns, pd.MultiIndex):
        btc_data.columns = btc_data.columns.droplevel(1)

    btc_data = btc_data[['Close']].dropna()  # Keep only the 'Close' price and drop NaN values
    return btc_data

def calculate_bollinger_bands(data, window=145, num_std_dev=2):
    """
    Adds columns for the Simple Moving Average (SMA), Upper Band, and Lower Band to the DataFrame.
    """
    data['SMA'] = data['Close'].rolling(window=window).mean()
    data['STD'] = data['Close'].rolling(window=window).std()
    
    # Calculate Bollinger Bands
    data['Upper_Band'] = data['SMA'] + (num_std_dev * data['STD'])
    data['Lower_Band'] = data['SMA'] - (num_std_dev * data['STD'])
    
    return data

def detect_bounces(data, sma_col='SMA', price_col='Close', threshold=0.01):
    """
    Detects bounces where price comes close to the SMA or within Bollinger Bands.
    """
    data['Distance'] = (data[price_col] - data[sma_col]) / data[sma_col]
    
    # Detect bounces near SMA within a threshold
    data['Bounce_SMA'] = ((data['Distance'].shift(1) < threshold) & 
                          (data['Distance'] > threshold)).astype(int)
    
    # Detect bounces off Bollinger Bands
    data['Bounce_Lower_Band'] = (data[price_col] <= data['Lower_Band']).astype(int)
    data['Bounce_Upper_Band'] = (data[price_col] >= data['Upper_Band']).astype(int)
    
    return data
# Step 4: Calculate statistics for bounces and distances
def calculate_statistics(data):
    """
    Calculates statistics for bounces off Bollinger Bands and distances.
    """
    # Calculate bounce rates
    lower_band_bounces = data['Bounce_Lower_Band'].sum()
    upper_band_bounces = data['Bounce_Upper_Band'].sum()
    total_bounces = lower_band_bounces + upper_band_bounces
    total_opportunities = len(data)

    lower_band_bounce_rate = (lower_band_bounces / total_opportunities) * 100
    upper_band_bounce_rate = (upper_band_bounces / total_opportunities) * 100
    overall_bounce_rate = (total_bounces / total_opportunities) * 100

    # Calculate distances for bounces
    lower_band_distances = data.loc[data['Bounce_Lower_Band'] == 1, 'Distance']
    upper_band_distances = data.loc[data['Bounce_Upper_Band'] == 1, 'Distance']

    distance_mean_lower = lower_band_distances.mean() if not lower_band_distances.empty else None
    distance_mean_upper = upper_band_distances.mean() if not upper_band_distances.empty else None

    distance_std_lower = lower_band_distances.std() if not lower_band_distances.empty else None
    distance_std_upper = upper_band_distances.std() if not upper_band_distances.empty else None

    return {
        'lower_band_bounce_rate': lower_band_bounce_rate,
        'upper_band_bounce_rate': upper_band_bounce_rate,
        'overall_bounce_rate': overall_bounce_rate,
        'distance_mean_lower': distance_mean_lower,
        'distance_mean_upper': distance_mean_upper,
        'distance_std_lower': distance_std_lower,
        'distance_std_upper': distance_std_upper
    }


def plot_btc_data(data):
    # Reset index to include it as a column in the DataFrame
    data = data.reset_index()
    # Prepare data for Bokeh
    # Ensure the 'index' column is a proper datetime object
    data['index'] = pd.to_datetime(data['Datetime'])
    source = ColumnDataSource(data)

    # Create x-coordinates for the patch (combine Upper_Band and reversed Lower_Band)
    x_patch = list(data['Datetime']) + list(data['Datetime'][::-1])
    y_patch = list(data['Upper_Band']) + list(data['Lower_Band'][::-1])

    # Create Bokeh figure
    p = figure(x_axis_type="datetime", title="Bitcoin Price with Bollinger Bands", width=800, height=400)
    p.grid.grid_line_alpha = 0.3

    # Plot Close price
    p.line(x='index', y='Close', source=source, color='blue', legend_label='Close Price', line_width=2)

    # Plot Upper and Lower Bands
    p.line(x='index', y='Upper_Band', source=source, color='orange', legend_label='Upper Band', line_width=2)
    p.line(x='index', y='Lower_Band', source=source, color='green', legend_label='Lower Band', line_width=2)

    # Add shaded area between Upper and Lower Bands
    p.patch(x=x_patch, y=y_patch, color="gray", alpha=0.3, legend_label="Bollinger Bands")

    # Customize plot
    p.legend.location = "top_left"
    p.xaxis.axis_label = "Datetime"
    p.yaxis.axis_label = "Price"

    # Show plot inline (for Jupyter Notebook) or in a browser
    output_notebook()  # Use this if running in a Jupyter Notebook
    show(p)


# Main execution flow
if __name__ == "__main__":
    # Fetch Bitcoin data
    btc_data = fetch_bitcoin_data()

    # Calculate SMA and detect bounces
    # Calculate SMA and Bollinger Bands
    btc_data = calculate_bollinger_bands(btc_data, window=30)
    
    # Handle cases where rolling mean introduces NaN values
    btc_data.dropna(inplace=True)
    
    # Detect bounces
    btc_data = detect_bounces(btc_data)

    # Plot the results
    plot_btc_data(btc_data)
    pprint(calculate_statistics(btc_data))


[*********************100%***********************]  1 of 1 completed


Loading BokehJS ...

{'distance_mean_lower': -0.01731627512056393,
 'distance_mean_upper': 0.022620499373117338,
 'distance_std_lower': 0.01171989068538427,
 'distance_std_upper': 0.01867711742184817,
 'lower_band_bounce_rate': 5.1838107387220225,
 'overall_bounce_rate': 12.71019366809695,
 'upper_band_bounce_rate': 7.526382929374928}


In [67]:
btc_data.index

DatetimeIndex(['2023-12-07 00:00:00+00:00', '2023-12-07 01:00:00+00:00',
               '2023-12-07 02:00:00+00:00', '2023-12-07 03:00:00+00:00',
               '2023-12-07 04:00:00+00:00', '2023-12-07 05:00:00+00:00',
               '2023-12-07 06:00:00+00:00', '2023-12-07 07:00:00+00:00',
               '2023-12-07 08:00:00+00:00', '2023-12-07 09:00:00+00:00',
               ...
               '2023-12-30 14:00:00+00:00', '2023-12-30 15:00:00+00:00',
               '2023-12-30 16:00:00+00:00', '2023-12-30 17:00:00+00:00',
               '2023-12-30 18:00:00+00:00', '2023-12-30 19:00:00+00:00',
               '2023-12-30 20:00:00+00:00', '2023-12-30 21:00:00+00:00',
               '2023-12-30 22:00:00+00:00', '2023-12-30 23:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='Datetime', length=576, freq=None)